# 🎓 Student Performance Analytics Dashboard
> **Theme:** Warm Cream &nbsp;|&nbsp; **Dataset:** Student Academic Performance &nbsp;|&nbsp; **Records:** 1,000 &nbsp;|&nbsp; **Cities:** 8 &nbsp;|&nbsp; **Subjects:** 6

---

| Step | Description |
|------|-------------|
| 1️⃣ | Install & Import Libraries |
| 2️⃣ | Load & Clean Data |
| 3️⃣ | KPI Summary Cards |
| 4️⃣ | Full Analytics Dashboard (9 Charts) |
| 5️⃣ | Download Output |

---

## 1️⃣ Install & Import Libraries

In [ ]:
!pip install matplotlib pandas openpyxl scipy --quiet

In [ ]:
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from matplotlib.patches import Rectangle, FancyBboxPatch
from matplotlib.colors import LinearSegmentedColormap
from scipy.stats import gaussian_kde
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline
plt.rcParams['figure.dpi'] = 130
plt.rcParams['font.family'] = 'DejaVu Sans'

print(f'✅  Matplotlib {matplotlib.__version__} | Pandas {pd.__version__}')

## 2️⃣ Load & Clean Data
> ⚠️ **Upload your file:** Run the cell below → file picker appears → select `Extra_Dataset_PowerBI.xlsx`

In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
FILE = 'Extra_Dataset_PowerBI.xlsx'
df_raw = pd.read_excel(FILE)
print(f'Rows: {df_raw.shape[0]}  |  Columns: {df_raw.shape[1]}')
df_raw.head(3)

In [ ]:
# ── Data Cleaning & Feature Engineering ──────────────────────────────────────
df = df_raw.copy()

# 1. Missing values
missing = df.isnull().sum()
print('Missing values:')
print(missing[missing > 0] if missing.sum() > 0 else '  ✅ None found')

# 2. Duplicates
before = len(df)
df.drop_duplicates(inplace=True)
print(f'Duplicates removed: {before - len(df)}')

# 3. Shorten verbose labels
subject_map = {
    'Advanced Physics':                 'Physics',
    'Botany and Zoology Biology':        'Biology',
    'Computer Science and Programming':  'CS',
    'English Literature and Grammar':    'English',
    'Organic and Inorganic Chemistry':   'Chemistry',
    'Pure and Applied Mathematics':      'Math'
}
df['Subject']  = df['Complete Subject Name'].map(subject_map)
df['City']     = df['Full City And State Name'].str.split().str[0]
df['Section']  = df['Section Name'].str.replace('Section ', '', regex=False)

# 4. Derived columns
df['Total Score'] = df['Midterm Examination Marks'] + df['Final Examination Marks']
df['Avg Score']   = df['Total Score'] / 2
df['Pass/Fail']   = np.where(df['Final Examination Marks'] >= 50, 'Pass', 'Fail')
df['Grade']       = pd.cut(
    df['Avg Score'],
    bins=[0, 40, 55, 70, 85, 100],
    labels=['F (<40)', 'D (40-55)', 'C (55-70)', 'B (70-85)', 'A (85+)']
)

print(f'\n✅  Final shape: {df.shape}')
df.head(3)

## 3️⃣ KPI Summary Cards

In [ ]:
# ── Cream Colour Palette (shared across all cells) ────────────────────────────
BG      = '#F5F0E8'   # warm cream background
CARD    = '#FFFDF7'   # soft white-cream card
CARD2   = '#FDF6EC'   # slightly deeper cream
BORDER  = '#E8DFD0'   # warm border
HEADER  = '#3D2B1F'   # deep espresso brown
TEXT    = '#2C1F14'   # dark brown
TEXT2   = '#6B5744'   # medium brown
TEXT3   = '#A08060'   # muted warm brown

C1 = '#C0392B'   # terracotta red
C2 = '#2980B9'   # warm blue
C3 = '#27AE60'   # sage green
C4 = '#E67E22'   # burnt orange
C5 = '#8E44AD'   # dusty purple
C6 = '#16A085'   # teal
C7 = '#D35400'   # deep orange
C8 = '#1ABC9C'   # mint

SUBJ_PAL  = [C2, C3, C1, C4, C5, C6]
CITY_PAL  = [C2, C3, C1, C4, C5, C6, C7, C8]
GRADE_CLR = {
    'A (85+)':   C3,
    'B (70-85)': C2,
    'C (55-70)': C4,
    'D (40-55)': C1,
    'F (<40)':   C5
}

# ── KPI Values ────────────────────────────────────────────────────────────────
total_students = len(df)
avg_attendance = df['Attendance Percentage'].mean()
avg_midterm    = df['Midterm Examination Marks'].mean()
avg_final      = df['Final Examination Marks'].mean()
pass_rate      = (df['Pass/Fail'] == 'Pass').mean() * 100
fail_rate      = 100 - pass_rate
top_subject    = df.groupby('Subject')['Avg Score'].mean().idxmax()
top_city       = df.groupby('City')['Avg Score'].mean().idxmax()

kpi_data = [
    ('TOTAL STUDENTS',  f'{total_students:,}',   C2,  '▲ Full Cohort'),
    ('AVG ATTENDANCE',  f'{avg_attendance:.1f}%', C3,  '▲ Above threshold'),
    ('AVG MIDTERM',     f'{avg_midterm:.1f}',     C4,  '/ 100 marks'),
    ('AVG FINAL',       f'{avg_final:.1f}',       C5,  '/ 100 marks'),
    ('PASS RATE',       f'{pass_rate:.1f}%',      C3,  f'Fail rate: {fail_rate:.1f}%'),
    ('TOP SUBJECT',     top_subject,              C1,  'Highest avg score'),
    ('TOP CITY',        top_city,                 C6,  'Highest avg score'),
    ('TOTAL SUBJECTS',  '6',                      C7,  '4 Sections covered'),
]

# ── Draw KPI Row ──────────────────────────────────────────────────────────────
fig_kpi, axes_kpi = plt.subplots(1, 8, figsize=(28, 2.6), facecolor=BG)
fig_kpi.subplots_adjust(left=0.01, right=0.99, wspace=0.20)
fig_kpi.suptitle('KEY PERFORMANCE INDICATORS', y=1.05,
                 color=TEXT3, fontsize=9, fontweight='bold')

for ax, (title, value, color, sub) in zip(axes_kpi, kpi_data):
    ax.set_facecolor(CARD)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_color(color); sp.set_linewidth(1.8)
    # Left accent bar
    ax.add_patch(Rectangle((0, 0), 0.055, 1,
                            transform=ax.transAxes, fc=color, lw=0))
    ax.text(0.55, 0.70, title, ha='center', va='center',
            fontsize=6.5, fontweight='bold', color=TEXT3,
            transform=ax.transAxes)
    ax.text(0.55, 0.42, value, ha='center', va='center',
            fontsize=16, fontweight='bold', color=color,
            transform=ax.transAxes)
    ax.text(0.55, 0.14, sub, ha='center', va='center',
            fontsize=7, color=TEXT2, transform=ax.transAxes)

plt.show()
print('✅  KPI cards rendered')

## 4️⃣ Full Analytics Dashboard — 9 Charts

| # | Chart Type | Insight |
|---|------------|---------|
| 1 | **Lollipop Chart** | Average Score by Subject |
| 2 | **KDE Area Chart** | Score Distribution: Midterm vs Final |
| 3 | **Donut Chart** | Gender Distribution |
| 4 | **Horizontal Bar** | Average Score by City |
| 5 | **Scatter Plot** | Attendance vs Final Score |
| 6 | **Stacked Bar** | Grade Distribution by Subject |
| 7 | **Violin Plot** | Score Spread by Subject |
| 8 | **Heatmap** | Final Score — City × Subject |
| 9 | **Ranked Bar** | Top 10 Students |

In [ ]:
# ── Helper ────────────────────────────────────────────────────────────────────
def style_ax(ax, title, xlabel='', ylabel=''):
    ax.set_facecolor(CARD)
    ax.set_title(title, color=HEADER, fontsize=10.5,
                 fontweight='bold', pad=10, loc='left')
    if xlabel: ax.set_xlabel(xlabel, color=TEXT3, fontsize=8.5, labelpad=5)
    if ylabel: ax.set_ylabel(ylabel, color=TEXT3, fontsize=8.5, labelpad=5)
    ax.tick_params(colors=TEXT2, labelsize=8.5, length=0)
    for sp in ax.spines.values():
        sp.set_color(BORDER); sp.set_linewidth(0.9)
    ax.set_axisbelow(True)
    ax.yaxis.grid(True,  color=BORDER, linewidth=0.7, linestyle='--', alpha=0.8)
    ax.xaxis.grid(False)

# ── Canvas ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(28, 20), facecolor=BG)

# Title band
fig.patches.append(Rectangle((0, 0.955), 1, 0.045,
                              transform=fig.transFigure, fc='#3D2B1F', lw=0, zorder=0))
fig.text(0.04, 0.978, '🎓  STUDENT PERFORMANCE ANALYTICS DASHBOARD',
         color='#F5F0E8', fontsize=18, fontweight='bold', va='center')
fig.text(0.04, 0.962,
         '  Academic Year  ·  1,000 Students  ·  6 Subjects  ·  8 Cities  ·  4 Sections',
         color='#C9B49A', fontsize=9, va='center')

# KPI mini-row inside the figure
for i, (title, value, color, sub) in enumerate(kpi_data):
    ax = fig.add_axes([0.03 + i * 0.1185, 0.875, 0.108, 0.072])
    ax.set_facecolor(CARD)
    ax.set_xlim(0, 1); ax.set_ylim(0, 1)
    ax.set_xticks([]); ax.set_yticks([])
    for sp in ax.spines.values():
        sp.set_color(color); sp.set_linewidth(1.8)
    ax.add_patch(Rectangle((0, 0), 0.055, 1,
                            transform=ax.transAxes, fc=color, lw=0))
    ax.text(0.55, 0.70, title, ha='center', va='center',
            fontsize=6.5, fontweight='bold', color=TEXT3, transform=ax.transAxes)
    ax.text(0.55, 0.42, value, ha='center', va='center',
            fontsize=16, fontweight='bold', color=color, transform=ax.transAxes)
    ax.text(0.55, 0.16, sub, ha='center', va='center',
            fontsize=7, color=TEXT2, transform=ax.transAxes)

gs = gridspec.GridSpec(3, 3, figure=fig,
                       left=0.04, right=0.97,
                       top=0.858, bottom=0.05,
                       hspace=0.50, wspace=0.32)

# ── Chart 1: Lollipop — Avg Score by Subject ──────────────────────────────────
ax1 = fig.add_subplot(gs[0, 0])
style_ax(ax1, '01  Avg Score by Subject', xlabel='Score (out of 100)')
ax1.yaxis.grid(False)
ax1.xaxis.grid(True, color=BORDER, linewidth=0.7, linestyle='--', alpha=0.8)
subj_avg = df.groupby('Subject')['Avg Score'].mean().sort_values()
y_pos = range(len(subj_avg))
ax1.hlines(y_pos, 0, subj_avg.values, colors=SUBJ_PAL[::-1], linewidth=2.5, alpha=0.75)
ax1.scatter(subj_avg.values, y_pos, color=SUBJ_PAL[::-1], s=110, zorder=5)
for i, val in enumerate(subj_avg.values):
    ax1.text(val + 0.8, i, f'{val:.1f}', va='center',
             fontsize=8.5, color=TEXT, fontweight='bold')
overall = df['Avg Score'].mean()
ax1.axvline(overall, color=C1, lw=1.5, ls='--', alpha=0.85)
ax1.text(overall + 0.4, len(subj_avg) - 0.35,
         f'μ = {overall:.1f}', color=C1, fontsize=7.5, fontweight='bold')
ax1.set_yticks(y_pos)
ax1.set_yticklabels(subj_avg.index, fontsize=9, color=TEXT)
ax1.set_xlim(0, 100)

# ── Chart 2: KDE Area — Score Distribution ────────────────────────────────────
ax2 = fig.add_subplot(gs[0, 1])
style_ax(ax2, '02  Score Distribution: Midterm vs Final',
         xlabel='Score', ylabel='Density')
for scores, color, label in [
    (df['Midterm Examination Marks'], C2, 'Midterm'),
    (df['Final Examination Marks'],   C1, 'Final')
]:
    kde = gaussian_kde(scores, bw_method=0.15)
    xs  = np.linspace(30, 105, 300)
    ys  = kde(xs)
    ax2.fill_between(xs, ys, alpha=0.22, color=color)
    ax2.plot(xs, ys, color=color, lw=2.2, label=label)
    ax2.axvline(scores.mean(), color=color, lw=1.5, ls='--', alpha=0.8)
    ax2.text(scores.mean() + 0.6, ax2.get_ylim()[1] * 0.92,
             f'μ={scores.mean():.1f}', color=color, fontsize=7.5)
ax2.legend(fontsize=8.5, labelcolor=TEXT, facecolor=CARD2,
           edgecolor=BORDER, framealpha=1)

# ── Chart 3: Donut — Gender ───────────────────────────────────────────────────
ax3 = fig.add_subplot(gs[0, 2])
ax3.set_facecolor(CARD)
for sp in ax3.spines.values(): sp.set_color(BORDER); sp.set_linewidth(0.9)
ax3.set_title('03  Gender Distribution', color=HEADER,
              fontsize=10.5, fontweight='bold', pad=10, loc='left')
gen = df['Gender'].value_counts()
gen_colors = {'Male': C2, 'Female': C1}
wedges, texts, autotexts = ax3.pie(
    gen.values, labels=gen.index, autopct='%1.1f%%',
    colors=[gen_colors[g] for g in gen.index],
    wedgeprops=dict(width=0.50, edgecolor=CARD, linewidth=3),
    startangle=90,
    textprops={'color': TEXT, 'fontsize': 9.5}
)
for at in autotexts:
    at.set_fontsize(9); at.set_color('white'); at.set_fontweight('bold')
ax3.text(0,  0.10, f'{len(df):,}', ha='center', va='center',
         fontsize=16, fontweight='bold', color=HEADER)
ax3.text(0, -0.20, 'Total', ha='center', fontsize=8, color=TEXT3)

# ── Chart 4: Horizontal Bar — Avg Score by City ───────────────────────────────
ax4 = fig.add_subplot(gs[1, 0])
style_ax(ax4, '04  Avg Score by City', xlabel='Average Score')
ax4.yaxis.grid(False)
ax4.xaxis.grid(True, color=BORDER, linewidth=0.7, linestyle='--', alpha=0.8)
city_avg = df.groupby('City')['Avg Score'].mean().sort_values()
bars4 = ax4.barh(city_avg.index, city_avg.values,
                 color=CITY_PAL[:len(city_avg)], height=0.55, edgecolor='none')
for bar, val in zip(bars4, city_avg.values):
    ax4.text(val + 0.3, bar.get_y() + bar.get_height() / 2,
             f'{val:.1f}', va='center', fontsize=8.5,
             color=TEXT, fontweight='bold')
ax4.axvline(df['Avg Score'].mean(), color=C7, lw=1.5, ls='--', alpha=0.85)
ax4.text(df['Avg Score'].mean() + 0.3, len(city_avg) - 0.35,
         f'μ={df["Avg Score"].mean():.1f}', color=C7, fontsize=7.5)
ax4.set_xlim(0, 90)
ax4.tick_params(axis='y', labelsize=9, colors=TEXT)

# ── Chart 5: Scatter — Attendance vs Final ────────────────────────────────────
ax5 = fig.add_subplot(gs[1, 1])
style_ax(ax5, '05  Attendance vs Final Score',
         xlabel='Attendance (%)', ylabel='Final Score')
pf_colors = {'Pass': C3, 'Fail': C1}
for pf, grp in df.groupby('Pass/Fail'):
    ax5.scatter(grp['Attendance Percentage'],
                grp['Final Examination Marks'],
                c=pf_colors[pf], alpha=0.35, s=18,
                label=pf, edgecolors='none')
z  = np.polyfit(df['Attendance Percentage'], df['Final Examination Marks'], 1)
xs = np.linspace(60, 100, 200)
ax5.plot(xs, np.poly1d(z)(xs), color=C7, lw=2.2, ls='--',
         label='Trend', zorder=5)
ax5.legend(fontsize=8.5, labelcolor=TEXT, facecolor=CARD2,
           edgecolor=BORDER, framealpha=1)

# ── Chart 6: Stacked Bar — Grade by Subject ───────────────────────────────────
ax6 = fig.add_subplot(gs[1, 2])
style_ax(ax6, '06  Grade Distribution by Subject', ylabel='Students')
grades_order = ['A (85+)', 'B (70-85)', 'C (55-70)', 'D (40-55)', 'F (<40)']
g_subj = df.groupby(['Subject', 'Grade']).size().unstack(fill_value=0)
g_subj = g_subj.reindex(
    columns=[g for g in grades_order if g in g_subj.columns]
)
bottom = np.zeros(len(g_subj))
for grade in g_subj.columns:
    vals = g_subj[grade].values
    ax6.bar(range(len(g_subj)), vals, bottom=bottom, label=grade,
            color=GRADE_CLR.get(grade, '#999'),
            edgecolor=CARD, linewidth=0.5, width=0.6)
    bottom += vals
ax6.set_xticks(range(len(g_subj)))
ax6.set_xticklabels(g_subj.index, rotation=35, ha='right',
                    fontsize=8.5, color=TEXT)
ax6.legend(fontsize=7.5, labelcolor=TEXT, facecolor=CARD2,
           edgecolor=BORDER, framealpha=1, ncol=1,
           loc='upper right', handlelength=1.2)

# ── Chart 7: Violin — Score Spread by Subject ─────────────────────────────────
ax7 = fig.add_subplot(gs[2, 0])
style_ax(ax7, '07  Score Spread by Subject (Violin)', ylabel='Final Score')
subjects_list = list(df['Subject'].unique())
data_vio = [
    df[df['Subject'] == s]['Final Examination Marks'].values
    for s in subjects_list
]
parts = ax7.violinplot(data_vio, positions=range(len(subjects_list)),
                       showmedians=True, showextrema=True, widths=0.7)
for pc, color in zip(parts['bodies'], SUBJ_PAL):
    pc.set_facecolor(color); pc.set_alpha(0.45); pc.set_edgecolor(color)
parts['cmedians'].set_color(HEADER); parts['cmedians'].set_linewidth(2.2)
parts['cmins'].set_color(TEXT3);    parts['cmaxes'].set_color(TEXT3)
parts['cbars'].set_color(TEXT3);    parts['cbars'].set_linewidth(1)
ax7.set_xticks(range(len(subjects_list)))
ax7.set_xticklabels(subjects_list, rotation=30, ha='right',
                    fontsize=8.5, color=TEXT)

# ── Chart 8: Heatmap — City × Subject ────────────────────────────────────────
ax8 = fig.add_subplot(gs[2, 1])
ax8.set_facecolor(CARD)
for sp in ax8.spines.values(): sp.set_color(BORDER); sp.set_linewidth(0.9)
ax8.set_title('08  Final Score — City × Subject', color=HEADER,
              fontsize=10.5, fontweight='bold', pad=10, loc='left')
heat = df.pivot_table(values='Final Examination Marks',
                      index='City', columns='Subject', aggfunc='mean')
cmap = LinearSegmentedColormap.from_list('cream_heat',
       ['#F5F0E8', '#E8A87C', '#C0392B'], N=256)
im = ax8.imshow(heat.values, cmap=cmap, aspect='auto', vmin=52, vmax=84)
ax8.set_xticks(range(len(heat.columns)))
ax8.set_yticks(range(len(heat.index)))
ax8.set_xticklabels(heat.columns, rotation=38, ha='right',
                    fontsize=8.5, color=TEXT2)
ax8.set_yticklabels(heat.index, fontsize=8.5, color=TEXT2)
for i in range(heat.shape[0]):
    for j in range(heat.shape[1]):
        v = heat.values[i, j]
        if not np.isnan(v):
            txt_color = 'white' if v > 70 else HEADER
            ax8.text(j, i, f'{v:.0f}', ha='center', va='center',
                     fontsize=8, color=txt_color, fontweight='bold')
cbar = fig.colorbar(im, ax=ax8, fraction=0.035, pad=0.03)
cbar.ax.tick_params(labelsize=7.5, colors=TEXT3)
cbar.outline.set_edgecolor(BORDER)

# ── Chart 9: Ranked Bar — Top 10 Students ─────────────────────────────────────
ax9 = fig.add_subplot(gs[2, 2])
style_ax(ax9, '09  Top 10 Students by Total Score', xlabel='Total Score')
ax9.yaxis.grid(False)
ax9.xaxis.grid(True, color=BORDER, linewidth=0.7, linestyle='--', alpha=0.8)
top10 = (df.nlargest(10, 'Total Score')
           [['Full Student Name', 'Total Score', 'Subject']]
           .reset_index(drop=True))
top10['Short'] = top10['Full Student Name'].str.split().str[-1]
y_pos = np.arange(len(top10))[::-1]
top_colors = [SUBJ_PAL[i % len(SUBJ_PAL)] for i in range(len(top10))]
bars_t = ax9.barh(y_pos, top10['Total Score'],
                  color=top_colors, height=0.62,
                  edgecolor='none', alpha=0.88)
ax9.set_yticks(y_pos)
ax9.set_yticklabels(
    [f"#{i+1}  {r['Short']}" for i, r in top10.iterrows()],
    fontsize=8.5, color=TEXT
)
for bar, val in zip(bars_t, top10['Total Score']):
    ax9.text(bar.get_width() + 0.6,
             bar.get_y() + bar.get_height() / 2,
             f'{val}', va='center', fontsize=8.5,
             color=TEXT, fontweight='bold')
ax9.set_xlim(0, 220)

# ── Footer strip ──────────────────────────────────────────────────────────────
fig.patches.append(Rectangle((0, 0), 1, 0.028,
                              transform=fig.transFigure,
                              fc='#3D2B1F', lw=0, zorder=0))
fig.text(0.04, 0.014,
         '✔  Data Cleaned & Validated  |  No Missing Values  |  1,000 Records',
         color='#C9B49A', fontsize=8)
fig.text(0.97, 0.014,
         'Built with Python  ·  Matplotlib  ·  Pandas',
         ha='right', color='#C9B49A', fontsize=8)

plt.savefig('Student_Dashboard_Cream.png', dpi=150,
            bbox_inches='tight', facecolor=BG)
plt.show()
print('✅  Dashboard saved as Student_Dashboard_Cream.png')

## 5️⃣ Download Dashboard

In [ ]:
from google.colab import files
files.download('Student_Dashboard_Cream.png')
print('📥  Download started!')